In [32]:
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, brier_score_loss, log_loss
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.api import Logit, add_constant


In [12]:
load_dotenv()

db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}")

In [13]:
# Helper function to sort rounds
def get_round_order(row):
    # Match regular rounds: 'R1', 'R23', etc.
    match = re.match(r'R(\d+)$', row['round'])
    if match:
        return int(match.group(1))
    # Finals rounds mapping (AFL convention)
    finals_order = {
        'REF': 100,  # Elimination Final (week 1)
        'RQF': 101,  # Qualifying Final (week 1)
        'RSF': 102,  # Semi Final (week 2)
        'RPF': 103,  # Preliminary Final (week 3)
        'RGF': 104,  # Grand Final (week 4)
    }
    code = row['round'][1:] 
    return finals_order.get(code, 999)  # Unknown finals get 999

In [14]:
# dataframe for games
df_games = pd.read_sql("SELECT * FROM games", engine)

# sort rounds
df_games['round_order'] = df_games.apply(get_round_order, axis=1)
df_games = df_games.sort_values(['season_year', 'round_order', 'game_date']).reset_index(drop=True)


In [17]:
# Create ELO
START_ELO = 1500
K = 40
HGA = 50
ELO_SOFT_RESET = 0.75

# Initialize ELO ratings
teams = pd.concat([df_games['home_team_id'], df_games['away_team_id']]).unique()
elo = {team: START_ELO for team in teams}
home_elo_history, away_elo_history = [], []

for idx, row in df_games.iterrows():
    # Soft reset if first round of a new season
    if idx > 0 and row['season_year'] != df_games.loc[idx-1, 'season_year']:
        for team in elo:
            elo[team] = elo[team] * ELO_SOFT_RESET + START_ELO * (1 - ELO_SOFT_RESET)

    home, away = row['home_team_id'], row['away_team_id']
    home_elo, away_elo = elo[home], elo[away]

    # Win prob with HGA
    exp_home = 1 / (1 + 10 ** ((away_elo - (home_elo + HGA)) / 400))

    # Actual result
    if row['home_score_total'] > row['away_score_total']:
        s_home, s_away = 1, 0
    elif row['home_score_total'] < row['away_score_total']:
        s_home, s_away = 0, 1
    else:
        s_home = s_away = 0.5

    # Store pre-game ELO for analysis/features
    home_elo_history.append(home_elo)
    away_elo_history.append(away_elo)

    # Update ELO
    elo[home] += K * (s_home - exp_home)
    elo[away] += K * (s_away - (1 - exp_home))

df_games['home_team_elo'] = home_elo_history
df_games['away_team_elo'] = away_elo_history


In [26]:
# Calculate ELO win probs
hga = 50 # home ground advantage
def elo_win_prob(home_elo, away_elo, hga=hga):
    return 1 / (1 + 10 ** ((away_elo - (home_elo + hga)) / 400))

df_games['elo_home_win_prob'] = elo_win_prob(
    df_games['home_team_elo'],
    df_games['away_team_elo']
)


In [27]:
# Calculate accuracy
df_games['elo_pred_home_win'] = df_games['elo_home_win_prob'] > 0.5
df_games['actual_home_win'] = df_games['home_score_total'] > df_games['away_score_total']

# Exclude the first season from evaluation
eval_mask = df_games['season_year'] > 2010
eval_games = df_games[eval_mask]

# Accuracy
accuracy = (eval_games['elo_pred_home_win'] == eval_games['actual_home_win']).mean()
print(f"ELO pick accuracy: {accuracy*100:.2f}%")

# Brier Score
brier = brier_score_loss(eval_games['actual_home_win'], eval_games['elo_home_win_prob'])
print(f"Brier Score: {brier:.4f}")

# Log Loss
logloss = log_loss(eval_games['actual_home_win'], eval_games['elo_home_win_prob'])
print(f"Log Loss: {logloss:.4f}")

# AUC score
auc = roc_auc_score(eval_games['actual_home_win'], eval_games['elo_home_win_prob'])
print(f"AUC: {auc:.4f}")


ELO pick accuracy: 67.88%
Brier Score: 0.2047
Log Loss: 0.5940
AUC: 0.7369


In [29]:
# Calculate regression
df_games['elo_diff'] = df_games['home_team_elo'] - df_games['away_team_elo']

# Exclude the first season
mask = df_games['season_year'] > 2010
X_lin = df_games.loc[mask, 'elo_diff']
y_lin = df_games.loc[mask, 'margin']

X_lin = sm.add_constant(X_lin)
model_lin = sm.OLS(y_lin, X_lin).fit()
print(model_lin.summary())
X_lin = df_games.loc[mask, 'elo_diff']
y_lin = df_games.loc[mask, 'margin']

X_lin = sm.add_constant(X_lin)
model_lin = sm.OLS(y_lin, X_lin).fit()
print(model_lin.summary())

                            OLS Regression Results                            
Dep. Variable:                 margin   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     32.98
Date:                Mon, 26 May 2025   Prob (F-statistic):           1.03e-08
Time:                        20:18:23   Log-Likelihood:                -13799.
No. Observations:                2930   AIC:                         2.760e+04
Df Residuals:                    2928   BIC:                         2.761e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         33.8563      0.496     68.202      0.0

In [33]:
# Logistic
y_log = (df_games.loc[mask, 'home_score_total'] > df_games.loc[mask, 'away_score_total']).astype(int)
X_log = df_games.loc[mask, 'elo_diff']
X_log = add_constant(X_log)

model_log = Logit(y_log, X_log).fit()
print(model_log.summary())

Optimization terminated successfully.
         Current function value: 0.593691
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                 2930
Model:                          Logit   Df Residuals:                     2928
Method:                           MLE   Df Model:                            1
Date:                Mon, 26 May 2025   Pseudo R-squ.:                  0.1334
Time:                        20:20:46   Log-Likelihood:                -1739.5
converged:                       True   LL-Null:                       -2007.2
Covariance Type:            nonrobust   LLR p-value:                1.837e-118
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.3035      0.041      7.381      0.000       0.223       0.384
elo_diff       0.0061      0.